In [3]:
import numpy as np
import pandas as pd
from datasets import Dataset, load_dataset

In [4]:
# =======================================================
# 1. 2-Bin Balanced Stream Sampling (Read vs. Unread)
# =======================================================
# Target: 50,000 total rows distributed evenly to eliminate bias
BIN_TARGETS = {
    "read": 25000,
    "unread": 25000
}

print("Step 1: Loading 'behavior' dataset in streaming mode...")
behavior_stream = load_dataset("liyucheng/goodreads", "behavior", streaming=True)

processed_splits = {}

for split_name, dataset_iterable in behavior_stream.items():
    print(f"\n--- Processing split: {split_name} ---")

    bin_counts = {"read": 0, "unread": 0}
    selected_rows = []

    for example in dataset_iterable:
        is_read = example.get("is_read")
        
        # Categorize the interaction
        category = "read" if is_read is True else "unread"

        # Collect the row if we haven't hit the limit for this category
        if bin_counts[category] < BIN_TARGETS[category]:
            bin_counts[category] += 1
            example["read_status"] = category
            selected_rows.append(example)

        # Terminate streaming once both bins reach their quota
        if bin_counts["read"] >= BIN_TARGETS["read"] and bin_counts["unread"] >= BIN_TARGETS["unread"]:
            print(f"Target of 50,000 rows reached for split '{split_name}'.")
            break

    print(f"Collected bin distribution for '{split_name}': {bin_counts}")

    if selected_rows:
        df_split = pd.DataFrame(selected_rows)
        processed_splits[split_name] = df_split
    else:
        print(f"Warning: No valid records collected for '{split_name}'.")

# =======================================================
# 2. Book Metadata Extraction & Author ID Resolution
# =======================================================
print("\nStep 2: Loading 'books' dataset...")
books_ds = load_dataset("liyucheng/goodreads", "books")
books_data = books_ds["train"] if "train" in books_ds else next(iter(books_ds.values()))
books_df = books_data.to_pandas()

def extract_primary_author(authors_val):
    """Extract the first/primary author_id from author dict or list."""
    if isinstance(authors_val, (list, np.ndarray)) and len(authors_val) > 0:
        first = authors_val[0]
        if isinstance(first, dict):
            return str(first.get("author_id", ""))
        return str(first)
    elif isinstance(authors_val, dict):
        return str(authors_val.get("author_id", ""))
    elif pd.notna(authors_val):
        return str(authors_val)
    return ""

if "authors" in books_df.columns:
    books_df["author_id"] = books_df["authors"].apply(extract_primary_author)
elif "author_id" in books_df.columns:
    books_df["author_id"] = books_df["author_id"].astype(str)

desired_book_cols = [
    "book_id", "title", "author_id", "num_pages",
    "publication_year", "genres", "description", "average_rating"
]
available_book_cols = [c for c in desired_book_cols if c in books_df.columns]
books_clean = books_df[available_book_cols].drop_duplicates(subset=["book_id"])
books_clean["book_id"] = books_clean["book_id"].astype(str)

# =======================================================
# 3. Final Merge and Export
# =======================================================
print("\nStep 3: Merging data and generating new baseline...")

for split_name, df_split in processed_splits.items():
    df_split["book_id"] = df_split["book_id"].astype(str)
    merged_df = pd.merge(df_split, books_clean, on="book_id", how="left")

    # Using a distinct filename to keep your old work perfectly safe
    output_file = "unbiased_book_recommender_50k.csv"
    merged_df.to_csv(output_file, index=False)

    print(f"\nCreated unbiased file: {output_file}")
    print(f"Total rows: {len(merged_df)}")
    print(f"is_read distribution:\n{merged_df['is_read'].value_counts(dropna=False)}")

print("Pipeline complete. New baseline dataset is ready.")

Step 1: Loading 'behavior' dataset in streaming mode...



--- Processing split: train ---
Target of 50,000 rows reached for split 'train'.
Collected bin distribution for 'train': {'read': 25000, 'unread': 25000}

Step 2: Loading 'books' dataset...


books/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  869MB            

books/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2360655 [00:00<?, ? examples/s]


Step 3: Merging data and generating new baseline...

Created unbiased file: unbiased_looknew_50k.csv
Total rows: 50000
is_read distribution:
is_read
True     25000
False    25000
Name: count, dtype: int64
Pipeline complete. New baseline dataset is ready.


In [5]:
# print("hi")

In [6]:
import pandas as pd
import numpy as np

# 1. Load the freshly generated balanced dataset
df = pd.read_csv('unbiased_book_recommender_50k.csv')

# 2. Convert date columns to datetime
date_cols = ['date_added', 'read_at']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# 3. Observation anchor date
observation_date = pd.to_datetime('2017-06-01')

# 4. Calculate elapsed days
df['days_since_added_to_observation'] = (observation_date - df['date_added']).dt.days
df['days_to_read'] = (df['read_at'] - df['date_added']).dt.days

# 5. Filter for eligible interactions (Must have been on shelf >= 90 days before observation cutoff)
# Books added < 90 days before the cutoff haven't had their full window yet (censored data)
df_eligible = df[df['days_since_added_to_observation'] >= 90].copy()

# 6. Define the true binary target:
# 1 = Read within 90 days
# 0 = Either never read (is_read == False) OR read after 90 days
condition_converted = (
    (df_eligible['is_read'] == True) & 
    (df_eligible['read_at'].notna()) & 
    (df_eligible['days_to_read'] >= 0) & 
    (df_eligible['days_to_read'] <= 90)
)

df_eligible['converted_within_target'] = np.where(condition_converted, 1, 0)

print(f"Eligible rows retained: {len(df_eligible)}")
print("\nTarget Class Distribution:")
print(df_eligible['converted_within_target'].value_counts())
print("\nTarget Class Proportion:")
print(df_eligible['converted_within_target'].value_counts(normalize=True) * 100)

# 7. Save the clean processed file
output_processed_v2 = 'processed_data_v2.csv'
df_eligible.to_csv(output_processed_v2, index=False)
print(f"\nSaved clean interaction dataset to: '{output_processed_v2}'")

Eligible rows retained: 41942

Target Class Distribution:
converted_within_target
0    36283
1     5659
Name: count, dtype: int64

Target Class Proportion:
converted_within_target
0    86.507558
1    13.492442
Name: proportion, dtype: float64

Saved clean interaction dataset to: 'processed_data_v2.csv'


Chronological Master Feature Matrix Created Successfully!
Total training rows: 41942
File saved to: 'looknew_master_feature_matrix_v2.csv'



,user_id,book_id,converted_within_target,overall_shelf_to_read_conversion,short_conversion_rate,medium_conversion_rate,long_conversion_rate,genre_conversion_rate,genre_preference,author_conversion_rate
0,0151a92d6ef1e6b28bb9c9b05777d79d,5107,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
1,0151a92d6ef1e6b28bb9c9b05777d79d,2657,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
2,0151a92d6ef1e6b28bb9c9b05777d79d,320,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
3,0151a92d6ef1e6b28bb9c9b05777d79d,7437,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
4,0151a92d6ef1e6b28bb9c9b05777d79d,412732,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
5,0151a92d6ef1e6b28bb9c9b05777d79d,14942,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
6,0151a92d6ef1e6b28bb9c9b05777d79d,56373,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
7,0151a92d6ef1e6b28bb9c9b05777d79d,7588,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
8,0151a92d6ef1e6b28bb9c9b05777d79d,360635,1,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
9,0151a92d6ef1e6b28bb9c9b05777d79d,6241137,0,0.111111,0.0,0.0,0.0,0.0,0.0,0.111111


In [8]:
# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split
# from sklearn.linear_model import LogisticRegression
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import classification_report, roc_auc_score
# import joblib

# # 1. Load the chronological feature matrix
# df = pd.read_csv('book_recommender_master_feature_matrix_v2.csv')

# # 2. Define Features and Target
# feature_cols = [
#     'overall_shelf_to_read_conversion', 'short_conversion_rate', 
#     'medium_conversion_rate', 'long_conversion_rate',
#     'genre_conversion_rate', 'genre_preference', 'author_conversion_rate'
# ]

# X = df[feature_cols]
# y = df['converted_within_target']

# # 3. Stratified split (80% train, 20% test)
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )

# # 4. Feature scaling
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# # 5. Train Logistic Regression
# # class_weight='balanced' accounts for the natural 86/14 class imbalance
# log_model = LogisticRegression(class_weight='balanced', random_state=42)
# log_model.fit(X_train_scaled, y_train)

# # 6. Predictions & Evaluation
# y_pred = log_model.predict(X_test_scaled)
# y_prob = log_model.predict_proba(X_test_scaled)[:, 1]

# print("--- Logistic Regression Model Performance ---")
# print(classification_report(y_test, y_pred))
# print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}\n")

# # 7. Extract Feature Weights (for Rule-Based Explanations)
# weights = pd.DataFrame({
#     'Feature': feature_cols,
#     'Coefficient (Weight)': log_model.coef_[0]
# }).sort_values(by='Coefficient (Weight)', ascending=False)

# print("--- Behavioral Feature Weights ---")
# display(weights)

# # 8. Save artifacts for the final ranking step
# joblib.dump(log_model, 'book_rec_logistic_model.pkl')
# joblib.dump(scaler, 'book_rec_scaler.pkl')
# print("\nLogistic Regression model and scaler saved successfully.")

--- Logistic Regression Model Performance ---
              precision    recall  f1-score   support

           0       0.94      0.84      0.89      7257
           1       0.38      0.64      0.48      1132

    accuracy                           0.81      8389
   macro avg       0.66      0.74      0.68      8389
weighted avg       0.86      0.81      0.83      8389

ROC-AUC Score: 0.8333

--- Behavioral Feature Weights ---


,Feature,Coefficient (Weight)
6,author_conversion_rate,0.812738
0,overall_shelf_to_read_conversion,0.812738
1,short_conversion_rate,0.121835
5,genre_preference,0.055191
4,genre_conversion_rate,0.029737
2,medium_conversion_rate,-0.255567
3,long_conversion_rate,-0.317599



Logistic Regression model and scaler saved successfully.
